# Training a Linear Regression Model

Linear Regression is often the first real supervised model built after a regression baseline. It learns a linear relationship between features and a continuous target, and it remains one of the most interpretable models in machine learning.

This lesson covers the concept, evaluation, coefficient interpretation, and how to fit the model correctly with a leakage-free pipeline.

## Why Linear Regression matters

Linear Regression is useful because it is simple, transparent, and often surprisingly strong when the relationship between features and target is approximately linear.

It is especially useful when you want:

- A strong interpretable baseline
- Coefficients with business meaning
- A regression model that is easy to debug
- A comparison point before trying more complex methods

In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(41)
n_rows = 600

df = pd.DataFrame({
    'size_sqft': rng.integers(500, 3500, size=n_rows),
    'bedrooms': rng.integers(1, 6, size=n_rows),
    'age_years': rng.integers(1, 40, size=n_rows),
})

df['price'] = (
    df['size_sqft'] * 1200
    + df['bedrooms'] * 15000
    - df['age_years'] * 1800
    + rng.normal(0, 25000, size=n_rows)
).round(2)

df.head()

## Train, baseline, and model

Always split before fitting anything. The baseline must be learned from the training set only, and the regression model should be evaluated on held-out test data.

A pipeline is useful even when the model itself does not strictly need scaling, because it keeps preprocessing and prediction behavior aligned.

In [ ]:
TARGET_COLUMN = 'price'
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_preds = baseline.predict(X_test)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])

pipeline.fit(X_train, y_train)
model_preds = pipeline.predict(X_test)

results = {
    'baseline_rmse': np.sqrt(mean_squared_error(y_test, baseline_preds)),
    'baseline_mae': mean_absolute_error(y_test, baseline_preds),
    'baseline_r2': r2_score(y_test, baseline_preds),
    'model_rmse': np.sqrt(mean_squared_error(y_test, model_preds)),
    'model_mae': mean_absolute_error(y_test, model_preds),
    'model_r2': r2_score(y_test, model_preds),
}

print({k: round(v, 3) for k, v in results.items()})

## Interpreting coefficients

Coefficients tell you how much the target changes for a one-unit change in a feature, holding all other features constant. If features are scaled, the coefficients become easier to compare.

The intercept is the model prediction when all features are zero after preprocessing.

In [ ]:
lr_model = pipeline.named_steps['model']
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', ascending=False)

print(f'Intercept: {lr_model.intercept_:.3f}')
print(coef_df.to_string(index=False))

cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
print(f'\nCV R2 scores: {np.round(cv_scores, 3)}')
print(f'Mean CV R2:   {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

## Practical checklist

- Split before fitting baseline, scaler, or model.
- Compare against a mean baseline on the same metrics.
- Use a pipeline if preprocessing is involved.
- Inspect coefficients only when feature scaling and multicollinearity are under control.
- Check cross-validation stability before claiming success.

## Bonus resources

- Scikit-learn LinearRegression Documentation
- Scikit-learn Linear Models User Guide